<a href="https://colab.research.google.com/github/yourusername/inventory_2022/blob/main/advanced_paper_filtering/notebooks/setfit_training_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SetFit Paper Classification Training

**Purpose**: Train SetFit few-shot learning model to classify papers as resource introductions vs usage papers

**Key Differences from Logistic Regression**:
- Deep learning (sentence embeddings)
- Few-shot learning (trains on title+abstract text)
- 10-20 minutes on GPU vs 3-5 hours on CPU
- Better capture of semantic patterns

**Target Performance**: ≥85% accuracy on borderline cases (score 0-2)

**Training Strategy**:
1. Load high-score (≥7) and low-score (≤-2) papers as training labels
2. Create balanced training set (20 positive, 20 negative)
3. Train SetFit on title+abstract embeddings
4. Classify medium-score papers (score 0-2)
5. Upload trained model to Google Drive

---

## Runtime Estimates
- **CPU**: ~3-5 hours
- **GPU (Colab T4)**: ~10-20 minutes ⭐

In [ ]:
# Generate session ID
import datetime
session_id = datetime.datetime.now().strftime('%Y-%m-%d-%H%M%S') + '_setfit_training'
print(f"Session ID: {session_id}")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ============================================
# CONFIGURATION
# ============================================

# Session configuration
SESSION_ID = session_id
BASE_PATH = '/content/drive/MyDrive'
REPO_PATH = os.path.join(BASE_PATH, 'inventory_2022/advanced_paper_filtering')

# Model configuration
MODEL_NAME = 'setfit_introduction_classifier'
BASE_MODEL = 'sentence-transformers/all-mpnet-base-v2'

# Training configuration
N_POSITIVE = 20  # Number of positive examples
N_NEGATIVE = 20  # Number of negative examples
BATCH_SIZE = 16
NUM_EPOCHS = 1

# Paths (FULL PATHS REQUIRED)
DATA_PATH = os.path.join(REPO_PATH, 'data')
HIGH_SCORE_FILE = os.path.join(DATA_PATH, 'results/high_score_papers.csv')
LOW_SCORE_FILE = os.path.join(DATA_PATH, 'results/low_score_papers.csv')
MEDIUM_SCORE_FILE = os.path.join(DATA_PATH, 'results/medium_score_papers.csv')
OUTPUT_PATH = os.path.join(REPO_PATH, 'models', SESSION_ID)

# Create output directory
os.makedirs(OUTPUT_PATH, exist_ok=True)

print(f"Configuration Complete")
print(f"SESSION_ID: {SESSION_ID}")
print(f"OUTPUT_PATH: {OUTPUT_PATH}")
print(f"\nExpected file paths:")
print(f"  High score papers: {HIGH_SCORE_FILE}")
print(f"  Low score papers: {LOW_SCORE_FILE}")
print(f"  Medium score papers: {MEDIUM_SCORE_FILE}")

In [ ]:
# Change to repository directory
os.chdir(REPO_PATH)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# System information
import sys
import torch

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("⚠️  WARNING: No GPU detected - training will be slow!")
    print("   Expected: 10-20 min on GPU")
    print("   On CPU: 3-5 hours")

In [ ]:
# Install SetFit and dependencies
print("Installing SetFit and dependencies...")
print("This will take 2-3 minutes")
!pip install setfit datasets tf-keras -q
print("✓ Installation complete")

In [ ]:
# Import libraries
import os

# CRITICAL: Disable WandB to avoid login prompt
os.environ['WANDB_DISABLED'] = 'true'

import pandas as pd
import numpy as np
import logging
import json
from datetime import datetime

from setfit import SetFitModel, Trainer, TrainingArguments
from datasets import Dataset

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("Libraries imported successfully")
print(f"SetFit version: {setfit.__version__}")

In [ ]:
# ============================================
# STEP 1: LOAD AND PREPARE TRAINING DATA
# ============================================

logger.info("="*80)
logger.info("STEP 1: CREATE TRAINING DATA")
logger.info("="*80)

# Load high-score papers (positives)
logger.info("Loading high-score papers...")
high_score_df = pd.read_csv(HIGH_SCORE_FILE)
logger.info(f"Loaded {len(high_score_df):,} high-score papers")

# Load low-score papers (negatives)
logger.info("Loading low-score papers...")
low_score_df = pd.read_csv(LOW_SCORE_FILE)
logger.info(f"Loaded {len(low_score_df):,} low-score papers")

# Create positive examples (high-score papers with score >= 7)
logger.info(f"Sampling {N_POSITIVE} positive examples (score >= 7)...")
positives = high_score_df[high_score_df['ling_score'] >= 7].sample(
    min(N_POSITIVE, len(high_score_df[high_score_df['ling_score'] >= 7])),
    random_state=42
).copy()
positives['text'] = positives['title'] + ' ' + positives['abstract']
positives['label'] = 1
logger.info(f"Created {len(positives)} positive examples")

# Create negative examples (low-score papers with score <= -2)
logger.info(f"Sampling {N_NEGATIVE} negative examples (score <= -2)...")
negatives = low_score_df[low_score_df['ling_score'] <= -2].sample(
    min(N_NEGATIVE, len(low_score_df[low_score_df['ling_score'] <= -2])),
    random_state=42
).copy()
negatives['text'] = negatives['title'] + ' ' + negatives['abstract']
negatives['label'] = 0
logger.info(f"Created {len(negatives)} negative examples")

# Combine and shuffle
training_data = pd.concat([positives, negatives], ignore_index=True)
training_data = training_data.sample(frac=1, random_state=42).reset_index(drop=True)

logger.info(f"\nTraining data created:")
logger.info(f"  Total examples: {len(training_data)}")
logger.info(f"  Positive (introductions): {(training_data['label'] == 1).sum()}")
logger.info(f"  Negative (usage): {(training_data['label'] == 0).sum()}")

# Save training data
training_file = os.path.join(OUTPUT_PATH, 'training_data.csv')
training_data.to_csv(training_file, index=False)
logger.info(f"\nTraining data saved to: {training_file}")

# Display sample examples
print("\n" + "="*80)
print("SAMPLE TRAINING EXAMPLES")
print("="*80)
print("\nPositive examples (introductions):")
for i, row in training_data[training_data['label'] == 1].head(3).iterrows():
    print(f"  - PMID {row['pmid']}: {row['text'][:100]}...")
    
print("\nNegative examples (usage):")
for i, row in training_data[training_data['label'] == 0].head(3).iterrows():
    print(f"  - PMID {row['pmid']}: {row['text'][:100]}...")
    
print("="*80 + "\n")

In [ ]:
# ============================================
# STEP 2: INITIALIZE SETFIT MODEL
# ============================================

logger.info("="*80)
logger.info("STEP 2: INITIALIZE SETFIT MODEL")
logger.info("="*80)

logger.info(f"Base model: {BASE_MODEL}")
logger.info("Loading model from HuggingFace...")

# Create SetFit model
model = SetFitModel.from_pretrained(
    BASE_MODEL,
    use_differentiable_head=True,
    head_params={"out_features": 2}
)

logger.info("✓ Model loaded successfully")

# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
logger.info(f"Training device: {device}")

if device == 'cpu':
    logger.warning("⚠️  Training on CPU - this will take 3-5 hours!")
    logger.warning("⚠️  For faster training, enable GPU in Colab: Runtime > Change runtime type > T4 GPU")
else:
    logger.info(f"✓ GPU detected: {torch.cuda.get_device_name(0)}")
    logger.info("✓ Expected training time: 10-20 minutes")

print("\n" + "="*80 + "\n")

In [ ]:
# ============================================
# STEP 3: PREPARE DATASET AND TRAINER
# ============================================

logger.info("="*80)
logger.info("STEP 3: PREPARE DATASET AND TRAINER")
logger.info("="*80)

# Prepare dataset for SetFit
logger.info("Converting to HuggingFace Dataset...")
train_dataset = Dataset.from_pandas(training_data[['text', 'label']])
logger.info(f"✓ Dataset created with {len(train_dataset)} examples")

# Training arguments
args = TrainingArguments(
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    evaluation_strategy="no",
    save_strategy="epoch",
    output_dir=os.path.join(OUTPUT_PATH, 'checkpoints'),
    report_to="none"
)

logger.info(f"Training configuration:")
logger.info(f"  Batch size: {BATCH_SIZE}")
logger.info(f"  Epochs: {NUM_EPOCHS}")
logger.info(f"  Device: {device}")

# Create trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
)

logger.info("✓ Trainer initialized")
print("\n" + "="*80 + "\n")

In [ ]:
# ============================================
# STEP 4: TRAIN MODEL
# ============================================

logger.info("="*80)
logger.info("STEP 4: TRAIN SETFIT MODEL")
logger.info("="*80)

logger.info(f"Training on {len(training_data)} examples...")
logger.info(f"Estimated time: {'10-20 minutes (GPU)' if device == 'cuda' else '3-5 hours (CPU)'}")
logger.info("")

# Record start time
start_time = datetime.now()
logger.info(f"Training started at: {start_time}")

# Train
trainer.train()

# Record end time
end_time = datetime.now()
duration = end_time - start_time
logger.info(f"\nTraining completed at: {end_time}")
logger.info(f"Total training time: {duration}")

logger.info("✓ Training complete!")
print("\n" + "="*80 + "\n")

In [ ]:
# ============================================
# STEP 5: TEST ON TRAINING EXAMPLES
# ============================================

logger.info("="*80)
logger.info("STEP 5: TEST MODEL ON TRAINING EXAMPLES")
logger.info("="*80)

# Test on sample training examples
test_texts = training_data['text'].head(10).tolist()
test_labels = training_data['label'].head(10).tolist()

logger.info("Running predictions on 10 sample examples...")
predictions = model.predict(test_texts)

print("\nPredictions on sample training examples:")
print("-" * 80)
correct = 0
for i, (text, true_label, pred_label) in enumerate(zip(test_texts, test_labels, predictions)):
    match = "✓" if true_label == pred_label else "✗"
    correct += (true_label == pred_label)
    print(f"{match} True: {true_label}, Pred: {pred_label} - {text[:80]}...")

accuracy = correct / len(test_labels)
logger.info(f"\nTraining accuracy (sample): {correct}/{len(test_labels)} ({100*accuracy:.1f}%)")
print("\n" + "="*80 + "\n")

In [ ]:
# ============================================
# STEP 6: SAVE MODEL
# ============================================

logger.info("="*80)
logger.info("STEP 6: SAVE MODEL")
logger.info("="*80)

model_dir = os.path.join(OUTPUT_PATH, MODEL_NAME)
logger.info(f"Saving model to: {model_dir}")
model.save_pretrained(model_dir)
logger.info("✓ Model saved successfully")

print(f"\nModel files:")
!ls -lh {model_dir}

print("\n" + "="*80 + "\n")

In [ ]:
# ============================================
# STEP 7: CLASSIFY MEDIUM-SCORE PAPERS
# ============================================

logger.info("="*80)
logger.info("STEP 7: CLASSIFY MEDIUM-SCORE PAPERS")
logger.info("="*80)

# Load medium-score papers
logger.info("Loading medium-score papers...")
medium_df = pd.read_csv(MEDIUM_SCORE_FILE)
logger.info(f"Loaded {len(medium_df):,} medium-score papers")

# CRITICAL FIX: Fill NaN values before concatenation
logger.info("Preparing texts for prediction...")
medium_df['title'] = medium_df['title'].fillna('')
medium_df['abstract'] = medium_df['abstract'].fillna('')
medium_df['text'] = medium_df['title'] + ' ' + medium_df['abstract']

# Filter out empty or invalid texts
valid_mask = medium_df['text'].str.strip().str.len() > 0
invalid_count = (~valid_mask).sum()
if invalid_count > 0:
    logger.warning(f"⚠️  Found {invalid_count} papers with no title/abstract - skipping")
    
medium_valid_df = medium_df[valid_mask].copy()
texts = medium_valid_df['text'].tolist()

logger.info(f"Valid papers for prediction: {len(texts):,}")

# Predict
logger.info(f"Running predictions on {len(texts):,} papers...")
logger.info("This may take 5-10 minutes...")

predict_start = datetime.now()
predictions = model.predict(texts)
predict_end = datetime.now()
predict_duration = predict_end - predict_start

logger.info(f"✓ Predictions complete in {predict_duration}")

# Convert predictions to numpy/list (handles both CPU and CUDA tensors)
if torch.is_tensor(predictions):
    predictions = predictions.cpu().numpy()
elif hasattr(predictions, 'tolist'):
    predictions = predictions.tolist()

# Add predictions to valid dataframe
medium_valid_df['setfit_label'] = predictions

# Get prediction probabilities
logger.info("Calculating prediction probabilities...")
probas = model.predict_proba(texts)

# Convert probabilities to numpy (handles CUDA tensors)
if torch.is_tensor(probas):
    probas = probas.cpu().numpy()

medium_valid_df['setfit_confidence'] = probas.max(axis=1)

# Split by prediction
setfit_introductions = medium_valid_df[medium_valid_df['setfit_label'] == 1].copy()
setfit_usage = medium_valid_df[medium_valid_df['setfit_label'] == 0].copy()

# Further split introductions by confidence
high_conf_intro = setfit_introductions[setfit_introductions['setfit_confidence'] >= 0.7]
low_conf_intro = setfit_introductions[setfit_introductions['setfit_confidence'] < 0.7]

logger.info(f"\nClassification results:")
logger.info(f"  Valid papers processed: {len(medium_valid_df):,}")
logger.info(f"  Papers skipped (no text): {invalid_count}")
logger.info(f"  SetFit Introductions (high confidence >=0.7): {len(high_conf_intro):,}")
logger.info(f"  SetFit Introductions (low confidence <0.7): {len(low_conf_intro):,}")
logger.info(f"  SetFit Usage: {len(setfit_usage):,}")

# Save results
intro_file = os.path.join(OUTPUT_PATH, 'setfit_classified_introductions.csv')
usage_file = os.path.join(OUTPUT_PATH, 'setfit_classified_usage.csv')

logger.info(f"\nSaving results...")
setfit_introductions.to_csv(intro_file, index=False)
logger.info(f"  Introductions: {intro_file}")

setfit_usage.to_csv(usage_file, index=False)
logger.info(f"  Usage papers: {usage_file}")

print("\n" + "="*80 + "\n")

In [ ]:
# ============================================
# STEP 8: EXPORT TRAINING SUMMARY
# ============================================

logger.info("="*80)
logger.info("STEP 8: EXPORT TRAINING SUMMARY")
logger.info("="*80)

# Create summary dictionary (convert numpy types to Python native types)
summary = {
    'session_id': SESSION_ID,
    'timestamp': datetime.now().isoformat(),
    'model_name': MODEL_NAME,
    'base_model': BASE_MODEL,
    'device': device,
    'training_samples': int(len(training_data)),
    'n_positive': int(N_POSITIVE),
    'n_negative': int(N_NEGATIVE),
    'batch_size': int(BATCH_SIZE),
    'num_epochs': int(NUM_EPOCHS),
    'training_time': str(duration),
    'training_accuracy_sample': float(accuracy),
    'medium_score_papers_total': int(len(medium_df)),
    'medium_score_papers_valid': int(len(medium_valid_df)),
    'medium_score_papers_skipped': int(invalid_count),
    'setfit_introductions': int(len(setfit_introductions)),
    'setfit_usage': int(len(setfit_usage)),
    'high_confidence_introductions': int(len(high_conf_intro)),
    'low_confidence_introductions': int(len(low_conf_intro)),
    'prediction_time': str(predict_duration)
}

# Save summary
summary_file = os.path.join(OUTPUT_PATH, 'training_summary.json')
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

logger.info(f"✓ Summary saved to: {summary_file}")
print("\n" + "="*80 + "\n")

In [ ]:
# ============================================
# STEP 9: VERIFY FILES SAVED TO GOOGLE DRIVE
# ============================================

logger.info("="*80)
logger.info("STEP 9: VERIFY FILES SAVED TO GOOGLE DRIVE")
logger.info("="*80)

logger.info(f"\nAll files are saved to Google Drive at:")
logger.info(f"{OUTPUT_PATH}")
logger.info("")

# List all files created
import os
import glob

logger.info("Files created:")
logger.info("-" * 80)

# Model directory
if os.path.exists(model_dir):
    model_size = sum(os.path.getsize(os.path.join(model_dir, f)) 
                     for f in os.listdir(model_dir) if os.path.isfile(os.path.join(model_dir, f)))
    logger.info(f"✓ Model: {MODEL_NAME}/ ({model_size / (1024*1024):.1f} MB)")
    
# Results files
result_files = [
    'training_data.csv',
    'setfit_classified_introductions.csv',
    'setfit_classified_usage.csv',
    'training_summary.json'
]

for filename in result_files:
    filepath = os.path.join(OUTPUT_PATH, filename)
    if os.path.exists(filepath):
        size_kb = os.path.getsize(filepath) / 1024
        logger.info(f"✓ {filename} ({size_kb:.1f} KB)")

logger.info("-" * 80)
logger.info(f"\n📁 Google Drive Path:")
logger.info(f"   MyDrive/inventory_2022/advanced_paper_filtering/models/{SESSION_ID}/")
logger.info("")
logger.info("✓ All files are already on Google Drive!")
logger.info("✓ Access them from your Drive after Colab disconnects")

print("\n" + "="*80 + "\n")

In [ ]:
# ============================================
# COMPLETION SUMMARY
# ============================================

print(f"""
╔══════════════════════════════════════════════════════════╗
║          SetFit Training Complete                        ║
╚══════════════════════════════════════════════════════════╝

Session ID: {SESSION_ID}
Device: {device}

Training Results:
- Training samples: {len(training_data)}
- Training time: {duration}
- Training accuracy (sample): {accuracy:.1%}

Classification Results:
- Medium-score papers (total): {len(medium_df):,}
- Medium-score papers (valid): {len(medium_valid_df):,}
- Medium-score papers (skipped): {invalid_count}
- SetFit Introductions: {len(setfit_introductions):,} ({100*len(setfit_introductions)/len(medium_valid_df):.1f}%)
  - High confidence (≥0.7): {len(high_conf_intro):,}
  - Low confidence (<0.7): {len(low_conf_intro):,}
- SetFit Usage: {len(setfit_usage):,} ({100*len(setfit_usage)/len(medium_valid_df):.1f}%)
- Prediction time: {predict_duration}

Output Files:
- Model: {MODEL_NAME}/
- Training data: training_data.csv
- Introductions: setfit_classified_introductions.csv
- Usage papers: setfit_classified_usage.csv
- Summary: training_summary.json

Output Location:
{OUTPUT_PATH}

Next Steps:
1. Download results from Google Drive
2. Compare SetFit vs Logistic Regression results
3. Manual validation of borderline cases

""")